In [ ]:
#Apple vs Samsung

In [1]:
!pip install scikit-learn nltk tensorflow

In [2]:
!pip install matplotlib
 

In [3]:
#libraties


import re, string, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Classification models/metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Regression models/metrics
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Neural net
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# VADER sentiment (assumes you downloaded vader_lexicon once via terminal)
from nltk.sentiment import SentimentIntensityAnalyzer



In [4]:
#Data

df = pd.read_csv("Downloads/Amazon_Unlocked_Mobile.csv")
print("Raw shape:", df.shape)

Raw shape: (413840, 6)


In [5]:
# Expected columns in this dataset
COL_TEXT = "Reviews"
COL_RATING = "Rating"
COL_PRODUCT = "Product Name"
COL_BRAND = "Brand Name"

work = df.copy()

# existence checks
for c in [COL_TEXT, COL_RATING, COL_PRODUCT]:
    if c not in work.columns:
        raise ValueError(f"Column '{c}' not found. Available columns: {list(work.columns)[:30]} ...")

# rename to standard
work = work.rename(columns={
    COL_TEXT: "text",
    COL_RATING: "rating",
    COL_PRODUCT: "product"
})
if COL_BRAND in work.columns:
    work = work.rename(columns={COL_BRAND: "brand"})
else:
    work["brand"] = np.nan

work = work[["product", "brand", "text", "rating"]].copy()
work["text"] = work["text"].astype(str)
work["rating"] = pd.to_numeric(work["rating"], errors="coerce")
work = work.dropna(subset=["rating", "product", "text"])

print("Cleaned shape:", work.shape)


Cleaned shape: (413840, 4)


In [6]:
# 2) FILTER PRODUCTS BY REVIEW COUNT

MIN_REVIEWS_PER_PRODUCT = 30
prod_counts = work["product"].value_counts()
keep_products = prod_counts[prod_counts >= MIN_REVIEWS_PER_PRODUCT].index
work_f = work[work["product"].isin(keep_products)].copy()

print("Products kept:", len(keep_products))
print("Filtered shape:", work_f.shape)


Products kept: 1798
Filtered shape: (390059, 4)


In [7]:
# 3) TEXT CLEANING

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\\S+|www\\S+", " ", s)
    s = s.encode("ascii", "ignore").decode()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\\s+", " ", s).strip()
    return s

work_f["text_clean"] = work_f["text"].apply(clean_text)

In [10]:
# 4) SENTIMENT (VADER)

sia = SentimentIntensityAnalyzer()
work_f["vader_compound"] = work_f["text_clean"].apply(lambda t: sia.polarity_scores(t)["compound"])
work_f["vader_pos"] = work_f["text_clean"].apply(lambda t: sia.polarity_scores(t)["pos"])
work_f["vader_neg"] = work_f["text_clean"].apply(lambda t: sia.polarity_scores(t)["neg"])

# Aggregate sentiment by product (optional table for slides)
sent_by_prod = work_f.groupby("product").agg(
    avg_rating=("rating", "mean"),
    n_reviews=("rating", "size"),
    avg_vader=("vader_compound", "mean")
).sort_values(["avg_rating", "avg_vader"], ascending=False)

print("\nTop 10 products by avg_rating then avg_vader:")
print(sent_by_prod.head(10))



Top 10 products by avg_rating then avg_vader:
                                                    avg_rating  n_reviews  \
product                                                                     
Huawei P8 GRA-UL00 16GB Titanium Grey, 5.2" Scr...    4.800000         35   
Apple Watch 42mm Stainless Steel Case with Mila...    4.772727         44   
ASUS ZenFone 2 ZE550ML 4G LTE Factory Unlocked ...    4.733333         45   
ASUS ZenFone 2 ZE550ML 4G LTE Factory Unlocked ...    4.733333         45   
OtterBox Defender Series Case & Holster for App...    4.729167         48   
OtterBox Defender Series Holster Belt Clip for ...    4.677419         31   
Huawei P9 EVA-L09 32GB 5.2 Inch 12 MP 4G LTE Fa...    4.676471         34   
Honor 8 Unlocked Phone 32GB, Sapphire Blue (US ...    4.666667         30   
Apple Watch Sport 42mm Space Gray Aluminum Case...    4.666667         33   
Huawei P9 EVA-L09 32GB 5.2 Inch 12 MP 4G LTE Fa...    4.657895         38   

                            

In [9]:
# PART A — CLASSIFICATION: Positive (4–5) vs Negative (1–3)

clf_df = work_f[work_f["rating"].isin([1, 2, 4, 5])].copy()
clf_df["label"] = (clf_df["rating"] >= 4).astype(int)

print("\nClassification rows:", clf_df.shape)
print("Label counts:\n", clf_df["label"].value_counts())

X_clf = clf_df["text_clean"]
y_clf = clf_df["label"]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

tfidf_clf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), min_df=2, stop_words="english")
X_train_tfidf_clf = tfidf_clf.fit_transform(X_train_clf)
X_test_tfidf_clf  = tfidf_clf.transform(X_test_clf)



Classification rows: (360371, 9)
Label counts:
 label
1    269915
0     90456
Name: count, dtype: int64


In [ ]:
# Baseline Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_tfidf_clf, y_train_clf)

proba = lr.predict_proba(X_test_tfidf_clf)[:, 1]
pred  = (proba >= 0.5).astype(int)

print("\nLogistic Regression")
print("Accuracy :", accuracy_score(y_test_clf, pred))
print("Precision:", precision_score(y_test_clf, pred))
print("Recall   :", recall_score(y_test_clf, pred))
print("F1       :", f1_score(y_test_clf, pred))
print("ROC-AUC  :", roc_auc_score(y_test_clf, proba))

# RandomForestClassifier
rf_clf = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
rf_clf.fit(X_train_tfidf_clf, y_train_clf)

rf_proba = rf_clf.predict_proba(X_test_tfidf_clf)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

print("\nRandom Forest (Classifier)")
print("RF Accuracy :", accuracy_score(y_test_clf, rf_pred))
print("RF ROC-AUC  :", roc_auc_score(y_test_clf, rf_proba))

# GradientBoostingClassifier (dense small)
tfidf_small = TfidfVectorizer(max_features=6000, ngram_range=(1,2), min_df=2, stop_words="english")
Xtr_s = tfidf_small.fit_transform(X_train_clf).toarray()
Xte_s = tfidf_small.transform(X_test_clf).toarray()

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(Xtr_s, y_train_clf)

gb_proba = gb_clf.predict_proba(Xte_s)[:, 1]
gb_pred  = (gb_proba >= 0.5).astype(int)

print("\nGradient Boosting (Classifier)")
print("GB Accuracy :", accuracy_score(y_test_clf, gb_pred))
print("GB ROC-AUC  :", roc_auc_score(y_test_clf, gb_proba))

# StackingClassifier (LR + RF)
estimators = [
    ("lr", LogisticRegression(max_iter=2000)),
    ("rf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42))
]
stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method="predict_proba",
    n_jobs=-1
)

stack_clf.fit(X_train_tfidf_clf, y_train_clf)
st_proba = stack_clf.predict_proba(X_test_tfidf_clf)[:, 1]
st_pred  = (st_proba >= 0.5).astype(int)

print("\nStacking (Classifier)")
print("Stack Accuracy:", accuracy_score(y_test_clf, st_pred))
print("Stack ROC-AUC :", roc_auc_score(y_test_clf, st_proba))





Logistic Regression
Accuracy : 0.946833159902879
Precision: 0.9578753241061972
Recall   : 0.9717508891523414
F1       : 0.9647632183908046
ROC-AUC  : 0.9809172510413582

Random Forest (Classifier)
RF Accuracy : 0.973735691987513
RF ROC-AUC  : 0.9924108250754755


In [2]:
%store df
%store work
%store work_f
%store clf_df


UsageError: Unknown variable 'df'


In [1]:
%store -r df
%store -r work
%store -r work_f
%store -r clf_df

no stored variable or alias df
no stored variable or alias work
no stored variable or alias work_f
no stored variable or alias clf_df


In [ ]:
# PART B — REGRESSION: Predict rating 1–5

reg_df = work_f.copy()
X_reg = reg_df["text_clean"]
y_reg = reg_df["rating"].astype(float)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

tfidf_reg = TfidfVectorizer(max_features=12000, ngram_range=(1,2), min_df=2, stop_words="english")
X_train_tfidf_reg = tfidf_reg.fit_transform(X_train_reg)
X_test_tfidf_reg  = tfidf_reg.transform(X_test_reg)

def reg_report(y_true, y_pred, name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.3f}  RMSE={rmse:.3f}  R^2={r2:.3f}")
    return mae, rmse, r2




In [ ]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42, min_samples_split=10)
dt.fit(X_train_tfidf_reg, y_train_reg)
dt_pred = dt.predict(X_test_tfidf_reg)
reg_report(y_test_reg, dt_pred, "DecisionTree")


In [ ]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=400,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_tfidf_reg, y_train_reg)
rf_pred = rf.predict(X_test_tfidf_reg)
reg_report(y_test_reg, rf_pred, "RandomForest")


In [ ]:
# Bagging Regressor (DT base)
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=80,
    random_state=42,
    n_jobs=-1
)
bag.fit(X_train_tfidf_reg, y_train_reg)
bag_pred = bag.predict(X_test_tfidf_reg)
reg_report(y_test_reg, bag_pred, "BaggingRegressor")

In [ ]:
# Gradient Boosting Regressor (dense small)
tfidf_reg_small = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=2, stop_words="english")
Xtr_reg_dense = tfidf_reg_small.fit_transform(X_train_reg).toarray()
Xte_reg_dense = tfidf_reg_small.transform(X_test_reg).toarray()

gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(Xtr_reg_dense, y_train_reg)
gbr_pred = gbr.predict(Xte_reg_dense)
reg_report(y_test_reg, gbr_pred, "GradientBoostingRegressor")


In [ ]:
# Stacking Regressor (RF + GBR) — use the SAME dense features as GBR
estimators_reg = [
    ("rf", RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)),
    ("gbr", GradientBoostingRegressor(random_state=42))
]
stack_reg = StackingRegressor(
    estimators=estimators_reg,
    final_estimator=RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    n_jobs=-1
)
stack_reg.fit(Xtr_reg_dense, y_train_reg)
stack_pred = stack_reg.predict(Xte_reg_dense)
reg_report(y_test_reg, stack_pred, "StackingRegressor")



In [ ]:

# Neural Net (dense TF-IDF)

tfidf_nn = TfidfVectorizer(max_features=12000, ngram_range=(1,2), min_df=2, stop_words="english")
Xtr_nn = tfidf_nn.fit_transform(X_train_reg).toarray()
Xte_nn = tfidf_nn.transform(X_test_reg).toarray()

tf.random.set_seed(42)

nn = keras.Sequential([
    layers.Input(shape=(Xtr_nn.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="linear")
])

nn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae", keras.metrics.RootMeanSquaredError(name="rmse")]
)

history = nn.fit(
    Xtr_nn, y_train_reg,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    verbose=0 
)

nn_pred = nn.predict(Xte_nn).ravel()
reg_report(y_test_reg, nn_pred, "NeuralNet")

# Plot training curve
plt.figure()
plt.plot(history.history["loss"], label="train_mse")
plt.plot(history.history["val_loss"], label="val_mse")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Neural Net Training vs Validation Loss")
plt.legend()
plt.show()



In [ ]:
# Residual plots (choose best_pred manually)

best_name = "RandomForest"
best_pred = rf_pred  # dt_pred, rf_pred, bag_pred, gbr_pred, stack_pred, nn_pred

residuals = y_test_reg.values - best_pred

plt.figure()
plt.scatter(best_pred, residuals, alpha=0.6)
plt.axhline(0)
plt.xlabel("Predicted rating")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Residuals vs Predicted ({best_name})")
plt.show()

plt.figure()
plt.hist(residuals, bins=30)
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Count")
plt.title(f"Residual Distribution ({best_name})")
plt.show()

plt.figure()
plt.scatter(y_test_reg, best_pred, alpha=0.6)
min_v = min(y_test_reg.min(), best_pred.min())
max_v = max(y_test_reg.max(), best_pred.max())
plt.plot([min_v, max_v], [min_v, max_v])
plt.xlabel("Actual rating")
plt.ylabel("Predicted rating")
plt.title(f"Predicted vs Actual ({best_name})")
plt.show()


In [ ]:
# Product ranking using RF predicted ratings + VADER + volume

all_X_tfidf = tfidf_reg.transform(work_f["text_clean"])
work_f["pred_rating_rf"] = rf.predict(all_X_tfidf)

agg = work_f.groupby("product").agg(
    avg_rating=("rating", "mean"),
    n_reviews=("rating", "size"),
    avg_vader=("vader_compound", "mean"),
    avg_pred_rating=("pred_rating_rf", "mean"),
).reset_index()

def minmax(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

agg["rating_norm"] = minmax(agg["avg_rating"])
agg["sent_norm"]   = minmax(agg["avg_vader"])
agg["pred_norm"]   = minmax(agg["avg_pred_rating"])
agg["vol_norm"]    = minmax(np.log1p(agg["n_reviews"]))

w_rating, w_sent, w_pred, w_vol = 0.45, 0.20, 0.25, 0.10
agg["quality_score"] = (
    w_rating * agg["rating_norm"] +
    w_sent   * agg["sent_norm"] +
    w_pred   * agg["pred_norm"] +
    w_vol    * agg["vol_norm"]
)

best = agg.sort_values("quality_score", ascending=False)
print("\nTop 15 products by quality score:")
print(best.head(15))

# Plot top 15
topn = 15
plot_df = best.head(topn).copy().sort_values("quality_score", ascending=True)

plt.figure(figsize=(10,6))
plt.barh(plot_df["product"], plot_df["quality_score"])
plt.xlabel("Quality score (0–1 weighted)")
plt.title(f"Top {topn} Phones by Composite Quality Score")
plt.tight_layout()
plt.show()
